# 1 Transformer Implementation

## Token embedding

In [1]:
import torch
import torch.nn as nn 
import math 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

vocab_size = 10000
d_model = 512
max_seq_len = 100 

# Token IDs
tokens = torch.tensor([
    [12, 45, 78,  0],
    [33, 17, 29, 91]
])

embedding = nn.Embedding(vocab_size,d_model)

x = embedding(tokens)

print("Token IDs shape :", tokens.shape)
print("Embedding shape :", x.shape)

Token IDs shape : torch.Size([2, 4])
Embedding shape : torch.Size([2, 4, 512])


## Positional encoding

In [2]:
position = torch.arange(max_seq_len).unsqueeze(1)

div_term = torch.exp(
    torch.arange(0, d_model, 2)
    * (-math.log(10000.0) / d_model)
)

pe = torch.zeros(max_seq_len, d_model)

pe[:, 0::2] = torch.sin(position * div_term)
pe[:, 1::2] = torch.cos(position * div_term)

# Only positions needed by our input
pe = pe[:tokens.size(1)]

# Add positional information
x = x + pe

print("After positional encoding:", x.shape)

After positional encoding: torch.Size([2, 4, 512])


## Multi-Head Attention

In [3]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0 

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model,d_model)
        self.W_k = nn.Linear(d_model,d_model)
        self.W_v = nn.Linear(d_model,d_model)

        self.W_o = nn.Linear(d_model,d_model)

    def forward(self, x, key_value = None, mask = None):

        batch_size,seq_len,_ = x.shape

        Q = self.W_q(x)

        if key_value is None:
            K = self.W_k(x)
            V = self.W_v(x) 

        else: 
            K = self.W_k(key_value)
            V = self.W_v(key_value)

        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1,2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1,2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1,2)

        scores = torch.matmul(
            Q,
            K.transpose(-1,-2)
        )

        scores = scores / math.sqrt(self.head_dim)

        # Optional Mask 
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights  = torch.softmax(scores, dim = -1)

        # Weighted Value 

        attention_output = torch.matmul(
            attention_weights,
            V
        )

        attention_output = attention_output.transpose(1,2)

        attention_output = attention_output.contiguous().view(
            batch_size,
            seq_len,
            self.d_model
        )

        output = self.W_o(attention_output)

        return output       

In [4]:
mha = MultiHeadAttention(
    d_model = 512,
    num_heads= 8
)

attention_output = mha(x) 

print(attention_output.shape)

torch.Size([2, 4, 512])


## Feed-Forward Network

In [5]:
class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.network(x)

In [6]:
ffn = FeedForward(
    d_model=512,
    d_ff=2048
)

ffn_output = ffn(attention_output)

print(ffn_output.shape)

torch.Size([2, 4, 512])


## Residual Connection + LayerNorm

```
              ┌───────────────┐
              │               │
X ───────────►│ Multi-Head    │
│             │ Attention     │
│             └───────┬───────┘
│                     │
└────────────── + ◄────┘
                 │
             LayerNorm
                 │
                 ▼
                 Y
                 │
                 ▼
               FFN
                 │
                 │
Y ─────────── + ─┘
                 │
             LayerNorm
                 │
                 ▼
                 Z

In [7]:
class EncoderBlock(nn.Module):

    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.attention = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.ffn = FeedForward(
            d_model,
            d_ff
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):

        # Self-attention + residual + normalization
        attention_output = self.attention(x, mask)

        x = self.norm1(
            x + attention_output
        )

        # Feed-forward + residual + normalization
        ffn_output = self.ffn(x)

        x = self.norm2(
            x + ffn_output
        )

        return x

In [8]:
encoder_block = EncoderBlock(
    d_model=512,
    num_heads=8,
    d_ff=2048
)

encoder_output = encoder_block(x)

print(encoder_output.shape)

torch.Size([2, 4, 512])


## Encoder

In [9]:
class Encoder(nn.Module):

    def __init__(self, num_layers, d_model, num_heads, d_ff):
        super().__init__()

        self.layers = nn.ModuleList([
            EncoderBlock(
                d_model,
                num_heads,
                d_ff
            )
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):

        for layer in self.layers:
            x = layer(x, mask)

        return x

In [10]:
encoder = Encoder(
    num_layers=2,
    d_model=512,
    num_heads=8,
    d_ff=2048
)

encoder_output = encoder(x)

print(encoder_output.shape)

torch.Size([2, 4, 512])


## Causal Mask

In [11]:
def create_causal_mask(seq_len):

    mask = torch.tril(
        torch.ones(
            seq_len,
            seq_len
        )
    )

    return mask

In [12]:
causal_mask = create_causal_mask(4)

print(causal_mask)

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


## Padding mask

In [13]:
def create_padding_mask(tokens, pad_token_id=0):
    """
    Create a mask that hides PAD tokens.

    tokens: (batch_size, seq_len)

    returns:
        (batch_size, 1, 1, seq_len)
    """
    mask = (tokens != pad_token_id)
    return mask.unsqueeze(1).unsqueeze(2)

In [14]:
src = torch.tensor([
    [12, 45, 78, 0],
    [33, 17, 29, 91]
])

src_padding_mask = create_padding_mask(src)

print(src_padding_mask)
print(src_padding_mask.shape)

tensor([[[[ True,  True,  True, False]]],


        [[[ True,  True,  True,  True]]]])
torch.Size([2, 1, 1, 4])


## Combine target padding + causal mask

In [15]:
def combine_masks(*masks):
    final_mask = masks[0]

    for mask in masks[1:]:
        final_mask = final_mask & mask

    return final_mask

In [16]:
tgt = torch.tensor([
    [5, 8, 11, 0],
    [7, 3, 14, 9]
])

tgt_padding_mask = create_padding_mask(tgt)

tgt_causal_mask = create_causal_mask(
    tgt.size(1),
    device=tgt.device
)

tgt_self_mask = combine_masks(
    tgt_padding_mask,
    tgt_causal_mask
)

print("Target padding:", tgt_padding_mask.shape)
print("Causal:", tgt_causal_mask.shape)
print("Target self:", tgt_self_mask.shape)

TypeError: create_causal_mask() got an unexpected keyword argument 'device'

## Decoder Block

In [ ]:
class DecoderBlock(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff
    ):
        super().__init__()

        # Masked self-attention
        self.self_attention = MultiHeadAttention(
            d_model,
            num_heads
        )

        # Cross-attention
        self.cross_attention = MultiHeadAttention(
            d_model,
            num_heads
        )

        # Feed-forward
        self.ffn = FeedForward(
            d_model,
            d_ff
        )

        # LayerNorm
        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.norm3 = nn.LayerNorm(
            d_model
        )

    def forward(
        self,
        x,
        encoder_output,
        self_mask=None,
        cross_mask=None
    ):

        # =================================
        # 1. Masked Self-Attention
        # =================================

        self_attention_output = self.self_attention(
            x,
            mask=self_mask
        )

        x = self.norm1(
            x + self_attention_output
        )

        # =================================
        # 2. Cross-Attention
        # =================================

        cross_attention_output = self.cross_attention(
            x,
            key_value=encoder_output,
            mask=cross_mask
        )

        x = self.norm2(
            x + cross_attention_output
        )

        # =================================
        # 3. Feed-Forward
        # =================================

        ffn_output = self.ffn(x)

        x = self.norm3(
            x + ffn_output
        )

        return x

## Test Decoder Block

In [ ]:
decoder_tokens = torch.tensor([
    [5, 8, 11, 0],
    [7, 3, 14, 9]
])

decoder_x = embedding(
    decoder_tokens
)

decoder_x = decoder_x + pe

causal_mask = create_causal_mask(
    decoder_x.size(1)
)

decoder_block = DecoderBlock(
    d_model=512,
    num_heads=8,
    d_ff=2048
)

decoder_output = decoder_block(
    decoder_x,
    encoder_output,
    self_mask=causal_mask
)

print("Decoder output:", decoder_output.shape)

## Decoder

In [ ]:
class Decoder(nn.Module):

    def __init__(
        self,
        num_layers,
        d_model,
        num_heads,
        d_ff
    ):
        super().__init__()

        self.layers = nn.ModuleList([

            DecoderBlock(
                d_model,
                num_heads,
                d_ff
            )

            for _ in range(num_layers)
        ])

    def forward(
        self,
        x,
        encoder_output,
        self_mask=None,
        cross_mask=None
    ):

        for layer in self.layers:

            x = layer(
                x,
                encoder_output,
                self_mask,
                cross_mask
            )

        return x

## Complete Transformer

In [ ]:
class Transformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_heads,
        d_ff,
        num_encoder_layers,
        num_decoder_layers,
        max_seq_len
    ):
        super().__init__()

        self.d_model = d_model

        # -----------------------------
        # Token Embedding
        # -----------------------------

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # -----------------------------
        # Positional Encoding
        # -----------------------------

        position = torch.arange(
            max_seq_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            )
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_seq_len,
            d_model
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        self.register_buffer(
            "pe",
            pe
        )

        # -----------------------------
        # Encoder
        # -----------------------------

        self.encoder = Encoder(
            num_encoder_layers,
            d_model,
            num_heads,
            d_ff
        )

        # -----------------------------
        # Decoder
        # -----------------------------

        self.decoder = Decoder(
            num_decoder_layers,
            d_model,
            num_heads,
            d_ff
        )

        # -----------------------------
        # Output Projection
        # -----------------------------

        self.output_projection = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(
        self,
        src,
        tgt
    ):

        # =================================
        # Encoder Input
        # =================================

        src_x = self.embedding(src)

        src_x = src_x + self.pe[
            :src.size(1)
        ]

        # =================================
        # Encoder
        # =================================

        encoder_output = self.encoder(
            src_x
        )

        # =================================
        # Decoder Input
        # =================================

        tgt_x = self.embedding(tgt)

        tgt_x = tgt_x + self.pe[
            :tgt.size(1)
        ]

        # =================================
        # Causal Mask
        # =================================

        tgt_mask = create_causal_mask(
            tgt.size(1)
        ).to(tgt.device)

        # =================================
        # Decoder
        # =================================

        decoder_output = self.decoder(
            tgt_x,
            encoder_output,
            self_mask=tgt_mask
        )

        # =================================
        # Vocabulary Prediction
        # =================================

        logits = self.output_projection(
            decoder_output
        )

        return logits

In [ ]:
model = Transformer(
    vocab_size=10000,
    d_model=512,
    num_heads=8,
    d_ff=2048,
    num_encoder_layers=2,
    num_decoder_layers=2,
    max_seq_len=100
)

print(model)

## Test the Complete Transformer

In [ ]:
# Source sequence
src = torch.tensor([
    [12, 45, 78,  0],
    [33, 17, 29, 91]
])

# Target sequence
tgt = torch.tensor([
    [5,  8, 11,  0],
    [7,  3, 14,  9]
])

# Forward pass
logits = model(
    src,
    tgt
)

print("Source shape :", src.shape)
print("Target shape :", tgt.shape)
print("Logits shape :", logits.shape)

## Check Parameter Count

In [ ]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Total parameters: {total_params:,}"
)

print(
    f"Trainable parameters: {trainable_params:,}"
)

In [ ]:
print("========== SHAPE CHECK ==========")

print("Source tokens       :", src.shape)
print("Target tokens       :", tgt.shape)

src_emb = model.embedding(src)
tgt_emb = model.embedding(tgt)

print("Source embeddings   :", src_emb.shape)
print("Target embeddings   :", tgt_emb.shape)

print("Encoder output      :", encoder_output.shape)
print("Final logits        :", logits.shape)

# 2 Masking